# 06 — News Intelligence & Sentiment

Phase 6 of the Indian Equity AI project plan.

Pipeline: news ingestion -> dedup/entity resolution -> FinBERT sentiment ->
event classification -> historical event-impact estimation -> daily (date, symbol)
news feature table.

**Data-source limitation:** Yahoo Finance's free news feed only returns a small,
recent rolling window per ticker (~10 items, last few days) -- not a multi-year
historical archive. The pipeline below is fully real (no synthetic/fabricated
news), but the event-impact statistics at the end are based on a small, recent
sample and should be read as a methodology demonstration, not a statistically
reliable estimate. A production system would need a licensed news provider
with historical coverage (see Indian_Equity_AI_Project_Plan.md Section 5).

In [1]:
# IMPORTANT: torch must be imported before pandas/pyarrow anywhere in this
# process, or its native DLL load fails on Windows (see src/news/sentiment.py
# module docstring for details). Keep this as the very first import.
import torch  # noqa: F401

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.news.ingestion import fetch_universe_news
from src.news.dedup import deduplicate_news, resolve_entity_mentions
from src.news.sentiment import add_sentiment, load_finbert_pipeline
from src.news.events import add_event_types
from src.news.impact import compute_event_occurrence_returns, explode_event_types, summarize_event_impact
from src.news.features import aggregate_daily_news_features

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
UNIVERSE = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ITC"]

## Step 1: News ingestion

In [3]:
news = fetch_universe_news(UNIVERSE)
print("Raw articles fetched:", len(news))
news["symbol"].value_counts()

Raw articles fetched: 41


symbol
RELIANCE    10
TCS         10
INFY        10
HDFCBANK    10
ITC          1
Name: count, dtype: int64

## Step 2: Deduplication and entity resolution

In [4]:
news = deduplicate_news(news)
news = resolve_entity_mentions(news)
print("After dedup:", len(news))
print("Genuinely mention the company:", news["mentions_company"].sum(), "/", len(news))

news = news[news["mentions_company"]].reset_index(drop=True)
news[["symbol", "headline", "published_timestamp"]]

After dedup: 41
Genuinely mention the company: 12 / 41


,symbol,headline,published_timestamp
0,RELIANCE,Reliance (NYSE:RS) Gets A Lift From Tax Relief...,2026-09-18 10:12:51+00:00
1,RELIANCE,Reliance’s JioHotstar takes its streaming empi...,2026-09-01 23:00:00+00:00
2,RELIANCE,Reliance's luxury unit brings Kim Kardashian's...,2026-08-05 12:13:21+00:00
3,RELIANCE,India's Reliance ramps up diesel exports to Eu...,2026-07-31 01:54:53+00:00
4,RELIANCE,Reliance Industries Ltd (WBO:RLI) Q1 2027 Earn...,2026-07-25 01:00:25+00:00
5,TCS,Tata Consultancy Services Stock And Other Indi...,2026-09-01 00:25:11+00:00
6,INFY,Chainlink’s Infosys Deal: 1.7 Billion Bank Acc...,2026-09-24 12:55:18+00:00
7,INFY,Infosys (INFY) Lands a Brake Maker as Growth S...,2026-09-19 19:47:47+00:00
8,INFY,Former Infosys chief’s AI startup nabs another...,2026-09-16 13:00:00+00:00
9,HDFCBANK,"India’s NPCI, HDFC tie up to launch sovereign ...",2026-09-10 11:57:23+00:00


## Step 3: FinBERT financial sentiment

In [5]:
finbert = load_finbert_pipeline()
news = add_sentiment(news, finbert)
news[["symbol", "headline", "positive_probability", "neutral_probability", "negative_probability"]]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,symbol,headline,positive_probability,neutral_probability,negative_probability
0,RELIANCE,Reliance (NYSE:RS) Gets A Lift From Tax Relief...,0.924008,0.018271,0.057722
1,RELIANCE,Reliance’s JioHotstar takes its streaming empi...,0.088835,0.901418,0.009747
2,RELIANCE,Reliance's luxury unit brings Kim Kardashian's...,0.923209,0.064435,0.012355
3,RELIANCE,India's Reliance ramps up diesel exports to Eu...,0.935926,0.039483,0.024591
4,RELIANCE,Reliance Industries Ltd (WBO:RLI) Q1 2027 Earn...,0.958474,0.020462,0.021064
5,TCS,Tata Consultancy Services Stock And Other Indi...,0.111194,0.872706,0.016100
6,INFY,Chainlink’s Infosys Deal: 1.7 Billion Bank Acc...,0.034038,0.243456,0.722505
7,INFY,Infosys (INFY) Lands a Brake Maker as Growth S...,0.324170,0.667817,0.008013
8,INFY,Former Infosys chief’s AI startup nabs another...,0.902147,0.079922,0.017931
9,HDFCBANK,"India’s NPCI, HDFC tie up to launch sovereign ...",0.073768,0.916393,0.009839


## Step 4: Event classification

In [6]:
news = add_event_types(news)
news[["symbol", "headline", "event_types"]]

,symbol,headline,event_types
0,RELIANCE,Reliance (NYSE:RS) Gets A Lift From Tax Relief...,[debt]
1,RELIANCE,Reliance’s JioHotstar takes its streaming empi...,[product_launch]
2,RELIANCE,Reliance's luxury unit brings Kim Kardashian's...,[]
3,RELIANCE,India's Reliance ramps up diesel exports to Eu...,[]
4,RELIANCE,Reliance Industries Ltd (WBO:RLI) Q1 2027 Earn...,[earnings]
5,TCS,Tata Consultancy Services Stock And Other Indi...,[dividend]
6,INFY,Chainlink’s Infosys Deal: 1.7 Billion Bank Acc...,[]
7,INFY,Infosys (INFY) Lands a Brake Maker as Growth S...,[]
8,INFY,Former Infosys chief’s AI startup nabs another...,"[product_launch, contract_win]"
9,HDFCBANK,"India’s NPCI, HDFC tie up to launch sovereign ...",[product_launch]


## Step 5: Historical event-impact analysis

How has each event type historically moved the stock at +1d / +5d / +20d?
(See caveat at the top of this notebook re: small sample size.)

In [7]:
prices = pd.read_parquet(DATA_PROCESSED / "prices_daily.parquet")[["date", "symbol", "close"]]

events_long = explode_event_types(news)
print("Event occurrences (news x event_type pairs):", len(events_long))

occurrence_returns = compute_event_occurrence_returns(events_long, prices, horizons=(1, 5, 20))
impact_summary = summarize_event_impact(occurrence_returns, horizons=(1, 5, 20))
impact_summary

Event occurrences (news x event_type pairs): 9


,event_type,occurrences,mean_return_1d,mean_return_5d,mean_return_20d
0,contract_win,1,-0.001321,-0.037264,NaN
1,debt,2,0.010919,-0.000349,-0.131497
2,dividend,1,-0.008864,-0.047911,NaN
3,earnings,1,-0.009609,0.030469,0.023281
4,management_change,1,-0.015592,-0.012502,NaN
5,product_launch,3,0.007546,-0.006787,NaN


## Step 6: Aggregate to a daily (date, symbol) news feature table

In [8]:
daily_news_features = aggregate_daily_news_features(news)
daily_news_features

,date,symbol,news_count,sentiment_mean,sentiment_std,positive_news_ratio,negative_news_ratio
0,2025-12-04,ITC,1,0.713069,NaN,1.0,0.0
1,2026-07-25,RELIANCE,1,0.937409,NaN,1.0,0.0
2,2026-07-31,RELIANCE,1,0.911336,NaN,1.0,0.0
3,2026-08-05,RELIANCE,1,0.910854,NaN,1.0,0.0
4,2026-09-01,HDFCBANK,1,-0.710076,NaN,0.0,1.0
5,2026-09-01,RELIANCE,1,0.079089,NaN,0.0,0.0
6,2026-09-01,TCS,1,0.095093,NaN,0.0,0.0
7,2026-09-10,HDFCBANK,1,0.063929,NaN,0.0,0.0
8,2026-09-16,INFY,1,0.884216,NaN,1.0,0.0
9,2026-09-18,RELIANCE,1,0.866286,NaN,1.0,0.0


## Step 7: Save

In [9]:
news.drop(columns=["event_types"]).assign(
    event_types=news["event_types"].apply(lambda types: ",".join(types))
).to_csv(DATA_PROCESSED / "news_scored.csv", index=False)

daily_news_features.to_parquet(DATA_PROCESSED / "news_daily_features.parquet", index=False)
impact_summary.to_csv(DATA_PROCESSED / "news_event_impact_summary.csv", index=False)

print("Saved:", DATA_PROCESSED / "news_scored.csv")
print("Saved:", DATA_PROCESSED / "news_daily_features.parquet")
print("Saved:", DATA_PROCESSED / "news_event_impact_summary.csv")

Saved: D:\Practise\Stock Market Analysis\data\processed\news_scored.csv
Saved: D:\Practise\Stock Market Analysis\data\processed\news_daily_features.parquet
Saved: D:\Practise\Stock Market Analysis\data\processed\news_event_impact_summary.csv
